# Частично видимые объекты

Данные соревнования и приватный clevr_fragments, две T4, интернет выключен. Каждый GPU обучает независимый seed. Метрики считаются на исходных JPEG официального val.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from scipy.special import expit
from sklearn.metrics import roc_auc_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Выберите GPU T4 x2"
ROOT = Path(tempfile.mkdtemp(prefix="clevr_verify_"))
RESULT = ROOT / "verify_result"
RESULT.mkdir()
print("Рабочая папка", ROOT, flush=True)

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError(name)
        z.extractall(dest)

def find_data(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / n).is_file() for n in ("val.csv","test.csv","sample_submission.csv"))
            and all((p.parent / n).is_dir() for n in ("train","val","test"))]

locations = list(INPUT.rglob("fragments_manifest.json"))
if not locations:
    archives = list(INPUT.rglob("clevr_fragments.zip"))
    if len(archives) != 1:
        raise ValueError("Подключите один Private Dataset из clevr_fragments.zip")
    unpack(archives[0], ROOT / "package")
    locations = [ROOT / "package/fragments_manifest.json"]
if len(locations) != 1:
    raise ValueError("Подключено несколько версий весов")
package = locations[0].parent
for key, digest in json.loads(locations[0].read_text()).items():
    with (package / key).open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise ValueError(key)
datasets = find_data(INPUT)
if not datasets:
    for j, path in enumerate(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            names = {Path(k).name for k in z.namelist()}
        if {"train.csv","val.csv","test.csv","sample_submission.csv"} <= names:
            unpack(path, ROOT / "inputs" / str(j))
    datasets = find_data(ROOT / "inputs")
if len(datasets) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = datasets[0]
env = dict(os.environ, CLEVR_DATA=str(DATA), CLEVR_PRIORS=str(package/"priors"),
           CLEVR_WEIGHTS=str(package/"weights_in"), PYTHONPATH=str(package),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

sys.path.insert(0,str(package))
sys.path.insert(0,str(package/'scripts/research'))

## Обучение на парных сценах и прогноз на val

In [ ]:
env.update(FRAGMENT_INPUT=str(package))
seeds=(421,853)
def run(seed,gpu,stage):
    directory=ROOT/f"seed_{seed}"
    directory.mkdir(exist_ok=True)
    done=directory/f"{stage}.done"
    if done.exists():
        return
    command=[sys.executable,"-m","scripts.research.fragment_votes",stage,str(seed)]
    logfile=WORK/f"fragments_{seed}_{stage}.log"
    with logfile.open("w") as log, subprocess.Popen(command,cwd=directory,
            env={**env,"CUDA_VISIBLE_DEVICES":str(gpu)},stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        for line in process.stdout:
            print(seed,line,end="",flush=True)
            log.write(line)
        if process.wait():
            raise RuntimeError(f"{stage} seed {seed}: {logfile}")
    done.touch()

def both(stage):
    with ThreadPoolExecutor(2) as pool:
        jobs=[pool.submit(run,s,g,stage) for g,s in enumerate(seeds)]
        for job in jobs:
            job.result()

both("fit")
both("val")

## Ошибки подсчёта на реальных сценах

In [ ]:
def outputs(split):
    frame=pd.read_csv(DATA/f"{split}.csv")
    def read(path):
        with np.load(path) as a:
            if not np.array_equal(a["ids"],frame.id) or not np.isfinite(a["score"]).all():
                raise ValueError(path)
            return a["score"]
    base=read(package/f"baseline_{split}.npz")
    core=frame.category.isin(["counting","spatial_counting","comparison"]).to_numpy()
    branches=[read(ROOT/f"seed_{s}/verify_result/control_{s}_{split}.npz") for s in seeds]
    def combine(branch):
        return np.where(core,.5*(base+branch),base)
    return frame,base,[combine(b) for b in branches],combine(np.mean(branches,axis=0))

frame,base,replicates,score=outputs("val")
y=frame.label.to_numpy()
def metrics(x):
    auc=roc_auc_score(y,x)
    return dict(auc=float(auc),wrong=int(((y==1)&(x<0)).sum()),tied=int(((y==1)&(x==0)).sum()),
                inversions=float(round((1-auc)*sum(y)*sum(1-y),1)))
summary={k:metrics(x) for k,x in zip(("baseline","seed421","seed853","fragments"),[base,*replicates,score])}
assert abs(summary["baseline"]["auc"]-.9997986666666666)<1e-8
display(pd.DataFrame(summary).T)
scene=pd.factorize(frame.image)[0]
n=len(np.unique(scene))
rng=np.random.default_rng(425)
delta=[]
for _ in range(1000):
    w=rng.multinomial(n,np.full(n,1/n))[scene]
    delta.append(roc_auc_score(y,score,sample_weight=w)-roc_auc_score(y,base,sample_weight=w))
ci=np.quantile(delta,[.025,.975])
passed=bool(ci[0]>0 and all(metrics(x)["auc"]>summary["baseline"]["auc"] for x in replicates)
            and summary["fragments"]["wrong"]+summary["fragments"]["tied"]<=summary["baseline"]["wrong"])
audit=frame.assign(base=base,fragments=score,corrected=(y==1)&(base<0)&(score>0),
                   regressed=(y==1)&(base>0)&(score<=0))
audit.to_csv(RESULT/"val_rows.csv",index=False)
display(audit[audit.label.eq(1)&((audit.base<=0)|(audit.fragments<=0))][
    ["image","caption","category","base","fragments","corrected","regressed"]].sort_values("base"))
# A wrong larger exact-number twin is an observable overcount on real scenes.
from scripts.research.native_centers import canonical_form,parse_program
overcounts={}
for key,values in (("base",base),("fragments",score)):
    over=under=0
    for _,g in frame[frame.category.isin(["counting","spatial_counting"])].groupby(["image","pair_id"]):
        pos=g[g.label==1].iloc[0]
        neg=g[g.label==0].iloc[0]
        if values[neg.name]<=values[pos.name]:
            continue
        number=lambda r:parse_program(canonical_form(r.caption,r.category)).number
        over+=int(number(neg)>number(pos))
        under+=int(number(neg)<number(pos))
    overcounts[key]=dict(over=over,under=under)
print("Точные количества на реальных сценах",overcounts)
report=dict(metrics=summary,delta_ci=ci.tolist(),eligible=passed,count_errors=overcounts)
(RESULT/"metrics.json").write_text(json.dumps(report,indent=2))
print("Интервал прироста",ci,"Переход к test",passed)

## Архив и условный прогноз на test

In [ ]:
if passed:
    both("test")
    frame,_,_,score=outputs("test")
    sample=pd.read_csv(DATA/"sample_submission.csv")
    assert frame.id.is_unique and sample.id.is_unique and set(frame.id)==set(sample.id)
    sample["label"]=pd.Series(expit(score),index=frame.id).loc[sample.id].to_numpy()
    path=WORK/"fragment_union.csv"
    sample.to_csv(path,index=False,float_format="%.17g")
    assert pd.read_csv(path,float_precision="round_trip").equals(sample) and sample.label.between(0,1).all()
with zipfile.ZipFile(WORK/"fragment_results.zip","w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
    for folder in [RESULT,*[ROOT/f"seed_{s}/verify_result" for s in seeds]]:
        for path in folder.glob("*"):
            archive.write(path,path.relative_to(ROOT))
    for path in WORK.glob("fragments_*.log"):
        archive.write(path,path.name)
os.chdir(WORK)
display(FileLink("fragment_results.zip"))
if passed:
    display(FileLink("fragment_union.csv"))